# General Script

In [1]:
%pip uninstall -y numpy pandas
%pip install numpy==1.26.4 pandas==2.2.3

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.1.3
Uninstalling pandas-2.1.3:
  Successfully uninstalled pandas-2.1.3
Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-2.2.3-cp39-cp39-win_amd64.whl.metadata (19 kB)
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ------------- -------------------------- 5.5/15.8 MB 56.4 MB/s eta 0:00:01
   ----------------------------- ---------- 11.5/15.8 MB 27.8 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 27.7 MB/s eta 0:00:00
Using cached pandas-2.2.3-cp39-cp39-win_amd64.whl (11.6 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: C:\Users\cathe\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np

print(pd.__version__)
print(np.__version__)

2.2.3
1.26.4


## Config

Three classes this semester: CS10, CS61A, CS61C. Each class has its own inputs/<course>/ folder of assignment CSVs and its own Group 1 / Group 2 study-roster CSVs under inputs/groups/.

In [2]:
from pathlib import Path

BASE_DIR = Path("C:/Users/cathe/Documents/Github/remind/Analysis")
INPUTS_DIR = BASE_DIR / "inputs"
GROUPS_DIR = INPUTS_DIR / "groups"
ANONYMIZED_DIR = BASE_DIR / "anonymized"

# Folder name under inputs/ for each class. Also used (uppercased) to find
# that class's study_<CLASS>_group_1.csv / _2.csv under inputs/groups/.
CLASSES = ["cs10", "cs61a", "cs61c"]

In [3]:
def normalize_email(email):
    return str(email).strip().lower()

### Load each class's Group 1 / Group 2 study rosters

Group CSVs live in inputs/groups/ and are named study_<CLASS>_group_<n>.csv (e.g. study_CS61A_group_1.csv) -- the class code in the filename is how a group list gets matched to its class.

In [4]:
def load_group_emails(course_code):
    """Return (group1_emails, group2_emails) for one class."""
    course = course_code.upper()
    group1_path = GROUPS_DIR / f"study_{course}_group_1.csv"
    group2_path = GROUPS_DIR / f"study_{course}_group_2.csv"
    group1_emails = set(pd.read_csv(group1_path)["email"].map(normalize_email))
    group2_emails = set(pd.read_csv(group2_path)["email"].map(normalize_email))
    return group1_emails, group2_emails

### Assign each student to Group 1, Group 2, or neither

In [5]:
def assign_group(email, group1_emails, group2_emails):
    email = normalize_email(email)
    if email in group1_emails:
        return "Group 1"
    if email in group2_emails:
        return "Group 2"
    return "None"

### Tag each class's assignment CSVs with Study Group, then anonymize

First Name, Last Name, and Email are dropped so the output csv is de-identified; SID is the primary key from here on. The anonymized copy is written to anonymized/<course_code>/, mirroring inputs/<course_code>/.

In [6]:
PII_COLUMNS = ["First Name", "Last Name", "Email"]


def process_class(course_code):
    """Load every assignment CSV for one class, tag each row with its study
    group, write an anonymized copy, and return {assignment_name: dataframe}.
    Safe to call on a class with no input files yet -- returns an empty dict.
    """
    group1_emails, group2_emails = load_group_emails(course_code)
    input_dir = INPUTS_DIR / course_code
    output_dir = ANONYMIZED_DIR / course_code
    output_dir.mkdir(parents=True, exist_ok=True)

    assignments = {}
    for csv_path in sorted(input_dir.glob("*.csv")):
        df = pd.read_csv(csv_path)
        df["Study Group"] = df["Email"].map(
            lambda email: assign_group(email, group1_emails, group2_emails)
        )
        df = df.drop(columns=PII_COLUMNS, errors="ignore")
        df.to_csv(output_dir / csv_path.name, index=False)
        assignments[csv_path.stem] = df
    return assignments

In [7]:
class_data = {course: process_class(course) for course in CLASSES}

for course, assignments in class_data.items():
    print(f"{course}: {len(assignments)} assignment file(s)")

cs10: 2 assignment file(s)
cs61a: 19 assignment file(s)
cs61c: 10 assignment file(s)


## Submission Category: Graded / Late / Missing

- Status == Missing -> **Missing**
- Status == Graded and Lateness (H:M:S) == 00:00:00 -> **Graded**
- Status == Graded and Lateness (H:M:S) != 00:00:00 -> **Late**

In [8]:
def classify_submission(status, lateness):
    status = str(status).strip().lower()
    if status == "missing":
        return "Missing"
    if status == "graded":
        return "Graded" if str(lateness).strip() == "00:00:00" else "Late"
    return "Other"


CATEGORIES = ["Graded", "Late", "Missing"]

### Build one summary table across every assignment, in every class

A class with no assignment files yet (e.g. CS10 / CS61C right now) simply contributes no rows -- nothing else needs to change once those folders fill in.

In [9]:
def submission_category_counts(class_data):
    """One row per (Class, Assignment, Study Group) with Graded/Late/Missing counts."""
    rows = []
    for course, assignments in class_data.items():
        for assignment_name, df in assignments.items():
            category = df.apply(
                lambda row: classify_submission(row["Status"], row["Lateness (H:M:S)"]),
                axis=1,
            )
            counts = (
                df.assign(**{"Submission Category": category})
                .groupby(["Study Group", "Submission Category"])
                .size()
                .unstack(fill_value=0)
            )
            for category_name in CATEGORIES:
                if category_name not in counts.columns:
                    counts[category_name] = 0
            counts = counts.reset_index()
            counts.insert(0, "Assignment", assignment_name)
            counts.insert(0, "Class", course)
            rows.append(counts)

    if not rows:
        return pd.DataFrame(columns=["Class", "Assignment", "Study Group", *CATEGORIES])

    combined = pd.concat(rows, ignore_index=True)
    other_cols = [
        c for c in combined.columns
        if c not in ["Class", "Assignment", "Study Group", *CATEGORIES]
    ]
    return combined[["Class", "Assignment", "Study Group", *CATEGORIES, *other_cols]]

In [10]:
submission_summary = submission_category_counts(class_data)
submission_summary

Submission Category,Class,Assignment,Study Group,Graded,Late,Missing,Other
0,cs10,[fa26] cs10 autoremind data - Project 1- Wordl...,Group 1,33,1,5,NaN
1,cs10,[fa26] cs10 autoremind data - Project 1- Wordl...,Group 2,34,4,2,NaN
2,cs10,[fa26] cs10 autoremind data - Project 1- Wordl...,None,64,26,45,NaN
3,cs10,[fa26] cs10 autoremind data - Project 2- Spell...,Group 1,22,3,14,NaN
4,cs10,[fa26] cs10 autoremind data - Project 2- Spell...,Group 2,29,1,10,NaN
...,...,...,...,...,...,...,...
88,cs61c,[fa26] cs61c autoremind data - Project 2B,Group 2,153,2,11,NaN
89,cs61c,[fa26] cs61c autoremind data - Project 2B,None,170,8,53,NaN
90,cs61c,[fa26] cs61c autoremind data - Project 3A,Group 1,153,3,9,NaN
91,cs61c,[fa26] cs61c autoremind data - Project 3A,Group 2,149,5,12,NaN


### Same breakdown, as rates within each (Class, Assignment, Study Group) group

In [11]:
def submission_category_rates(counts_df):
    rates = counts_df.copy()
    total = rates[CATEGORIES].sum(axis=1)
    for category_name in CATEGORIES:
        rates[f"{category_name} Rate"] = rates[category_name] / total
    return rates

pd.set_option('display.max_rows', None)
submission_rates = submission_category_rates(submission_summary)
submission_rates[submission_rates['Study Group'].isin(['Group 1', 'Group 2'])]


Submission Category,Class,Assignment,Study Group,Graded,Late,Missing,Other,Graded Rate,Late Rate,Missing Rate
0,cs10,[fa26] cs10 autoremind data - Project 1- Wordl...,Group 1,33,1,5,NaN,0.846154,0.025641,0.128205
1,cs10,[fa26] cs10 autoremind data - Project 1- Wordl...,Group 2,34,4,2,NaN,0.850000,0.100000,0.050000
3,cs10,[fa26] cs10 autoremind data - Project 2- Spell...,Group 1,22,3,14,NaN,0.564103,0.076923,0.358974
4,cs10,[fa26] cs10 autoremind data - Project 2- Spell...,Group 2,29,1,10,NaN,0.725000,0.025000,0.250000
6,cs61a,[fa26] cs61a autoremind data - Cats Checkpoint,Group 1,374,30,30,NaN,0.861751,0.069124,0.069124
7,cs61a,[fa26] cs61a autoremind data - Cats Checkpoint,Group 2,395,16,11,NaN,0.936019,0.037915,0.026066
9,cs61a,[fa26] cs61a autoremind data - Cats,Group 1,415,11,8,NaN,0.956221,0.025346,0.018433
10,cs61a,[fa26] cs61a autoremind data - Cats,Group 2,412,7,3,NaN,0.976303,0.016588,0.007109
12,cs61a,[fa26] cs61a autoremind data - Hog Checkpoint,Group 1,417,26,15,NaN,0.910480,0.056769,0.032751
13,cs61a,[fa26] cs61a autoremind data - Hog Checkpoint,Group 2,431,18,5,NaN,0.949339,0.039648,0.011013


In [12]:
submission_rates_merged = submission_rates.groupby(['Class', 'Assignment'], as_index=False).sum(numeric_only=True)
submission_rates_merged = submission_rates.set_index(['Class', 'Assignment', 'Study Group'])
submission_rates_merged

Submission Category                                                   Graded  \
Class Assignment                                         Study Group           
cs10  [fa26] cs10 autoremind data - Project 1- Wordle... Group 1          33   
                                                         Group 2          34   
                                                         None             64   
      [fa26] cs10 autoremind data - Project 2- Spelli... Group 1          22   
                                                         Group 2          29   
                                                         None             57   
cs61a [fa26] cs61a autoremind data - Cats Checkpoint     Group 1         374   
                                                         Group 2         395   
                                                         None            546   
      [fa26] cs61a autoremind data - Cats                Group 1         415   
                                                         Group 2         412   
                                                         None            582   
      [fa26] cs61a autoremind data - Hog Checkpoint      Group 1         417   
                                                         Group 2         431   
                                                         None            584   
      [fa26] cs61a autoremind data - Hog                 Group 1         431   
                                                         Group 2         418   
                                                         None            603   
      [fa26] cs61a autoremind data - Homework 1 (test... Group 1           0   
                                                         Group 2           0   
                                                         None              2   
      [fa26] cs61a autoremind data - Homework 1          Group 1         436   
                                                         Group 2         452   
                                                         None            652   
      [fa26] cs61a autoremind data - Homework 2          Group 1         441   
                                                         Group 2         445   
                                                         None            621   
      [fa26] cs61a autoremind data - Homework 3          Group 1         408   
                                                         Group 2         412   
                                                         None            573   
      [fa26] cs61a autoremind data - Homework 4          Group 1         394   
                                                         Group 2         391   
                                                         None            547   
      [fa26] cs61a autoremind data - Homework 5          Group 1         406   
                                                         Group 2         403   
                                                         None            566   
      [fa26] cs61a autoremind data - Homework 6          Group 1         404   
                                                         Group 2         402   
                                                         None            556   
      [fa26] cs61a autoremind data - Lab 0               Group 1         451   
                                                         Group 2         455   
                                                         None            703   
      [fa26] cs61a autoremind data - Lab 1               Group 1         434   
                                                         Group 2         442   
                                                         None            646   
      [fa26] cs61a autoremind data - Lab 2               Group 1         448   
                                                         Group 2         454   
                                                         None            596   
